In [ ]:
from dataclasses import dataclass
import matplotlib.pyplot as plt
import overcomplete_learning.em     as ol_em
import overcomplete_learning.data   as ol_data
import overcomplete_learning.metrics    as ol_metric
import overcomplete_learning.plotting    as ol_plot
import numpy                        as np
import ipympl

ModuleNotFoundError: No module named 'overcomplete_learning'

# Purpose
This notebook shows how we can visualize estimation of system for the proposed methods. Runs SB-VEM, LS-BS-VEM, LS-BS-VEM+, OverICA and Random. Set the settings below and press run all


In [ ]:
@dataclass
class StudyConfig:
    #data setup
    nreps                                   : int   = 2500      #sample size
    nobs                                    : int   = 3         #dim of X
    nsrc                                    : int   = 6        #dim of S
    #error options  
    err_sd                                  : float = 1         #scaling of error. Only works as place holder.
    sn_ratio                                : float = 500       #<1 low quality, >1 high quality
    correlation_type                        : str   = 'iid'     #Keep at i.i.d. error for now
    
    seed                                    : int   = 2         #error seed for reproducibility
    
    #EM options
    EM_iters                                : int   = 100      #The ELBO always increases, but not by a lot.
    convergence_threshold                   : float = 1e-4      #provide slightly earlier stopping to avoid too long simulation times
    n_inits                                 : int   = 1        #initializations of random parameters
    post_cov_scaling                        : float = 1         #1 = correct scaling, 0 = better empirical results     
    method                                  : str   = 'standard_iid' #estimation method. See run_study.py for how it works
    known_src_step                          : int   = 1
    normalize_A                             : bool  = True

#initializing 
cfg = StudyConfig()
cfg.seed =1
#setting noise sd.to be according to sn_noise_ration
cfg.err_sd = np.sqrt((cfg.nsrc*2)/(cfg.nobs*cfg.sn_ratio))
n_known_src_array = list(range(0, cfg.nsrc+1, cfg.known_src_step))
rng = np.random.default_rng(cfg.seed)
data = ol_data.generate_data(nreps = cfg.nreps, nobs=cfg.nobs, nsrc = cfg.nsrc, err_sd=cfg.err_sd, rng= rng, S_scale=1, dd1 = 0.99)


In [ ]:

def viz_2d_vector_scatterplot(data, results, method, cfg):
    A_true = ol_data.normalize_columns(data.A)
    arrow_scale = 5
    ncols = 3
    nrows = int(np.ceil((cfg.nsrc + 1) / ncols))
    dsrc = A_true.shape[1]
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(5 * ncols, 5 * nrows),
        sharex=True,
        sharey=True
    )

    # Flatten for easy indexing
    axes = np.array(axes).flatten()

    # Loop over number of known sources
    for n_known in range(cfg.nsrc + 1):
        ax = axes[n_known]
        # ---------------------------------------------
        # Construct known-source mask
        # ---------------------------------------------

        # Run EM
        out = results[n_known]

        # Align estimated dictionary
        
        A_est, _, _, _, _ = ol_data.best_permutation_match_sign_flips(
            A=A_true,
            B=out['A_est']
        )
        err = ol_metric.frobenius_err(true = A_true, estimate=A_est)[0]

        A_est_plot = ol_data.normalize_columns(A_est) * arrow_scale
        A_true_plot = A_true *arrow_scale
        # Scatter plot of observations
        ax.scatter( data.X[:, 0], data.X[:, 1], alpha=0.1, s=10)

        # Plot dictionary atoms
        for i in range(cfg.nsrc):
            # Highlight known sources
            alpha = 0.5 if i < n_known else 0.9
            col_true = 'tab:green' if i>=n_known else 'tab:cyan'
            col_est = 'tab:red' if i>=n_known else 'tab:cyan'
            # True atom
            ax.arrow(
                0,
                0,
                A_true_plot[0, i],
                A_true_plot[1, i],
                width=0.05,
                alpha=alpha,
                color=col_true
            )

            # Estimated atom
            ax.arrow( 0, 0, A_est_plot[0, i], A_est_plot[1, i], width=0.08, alpha=alpha, color=col_est, ls = ':')
        # Formatting
        ax.set_title(f"n_known = {n_known}, n_unknown = {dsrc - n_known}, err = {np.round(err,2)}, sd_err = {cfg.err_sd}")

        ax.set_aspect('equal', adjustable='box')
        ax.grid(True)

    # -------------------------------------------------
    # Remove unused axes
    # -------------------------------------------------
    for j in range(cfg.nsrc + 1, len(axes)):
        fig.delaxes(axes[j])

    # -------------------------------------------------
    # Global legend
    # -------------------------------------------------
    handles = [
        plt.Line2D([0], [0], color='tab:green', lw=3, label='True A'),
        plt.Line2D([0], [0], color='tab:red', lw=3, label='Estimated A')
    ]

    plt.tight_layout(rect=[0, 0, 1, 0.95])

    fig.legend(
        handles=handles,
        loc='upper center',
        ncol=2,
        fontsize=12
    )

    # -------------------------------------------------
    # Labels
    # -------------------------------------------------
    fig.suptitle(
        f"True vs Estimated Mixing Directions. Method: {method}",
        fontsize=16
    )

    plt.show()



In [ ]:
#RUN ALL METHODS ON THE DATA TO VISUALIZE BELOW

results_EM_iid = {}
results_EM_OLS_iid = {}
results_EM_OLS_debias_iid = {}
results_sdp = {}
results_fast_ICA = {}
results_EM_simple_wrong = {}
results_EM_random = {}

for n_known in range(cfg.nsrc + 1):
    print(f'n_known = {n_known}')
    sknown = ol_data.make_sknown(
        S=data.S,
        n_known_src=n_known, noise_level=0,
        rng = np.random.default_rng(cfg.seed))
     
    results_sdp[n_known] = ol_em.run_EM_OverICA(
        X=data.X,
        rng = np.random.default_rng(cfg.seed),
        S_known=sknown,
        n_known=n_known,
        EM_iter=cfg.EM_iters,
        whiten_data = False,
        scale_source = False,
        mu = 5, err_sd=cfg.err_sd
    )
    
    results_EM_iid[n_known] = ol_em.run_EM_extended_iid()(
        X=data.X,
        rng = np.random.default_rng(cfg.seed),
        EM_iter=cfg.EM_iters,
        err_tolerance=cfg.convergence_threshold,
        S_known=sknown,
        n_known=n_known,
        normalize_A_col=cfg.normalize_A,
        whiten_data = False,
        scale_source = False
    )

    results_EM_OLS_iid[n_known] = ol_em.run_EM_OLS_iid()(
        X=data.X,
        S_known=sknown,
        rng = np.random.default_rng(cfg.seed),
        EM_iter=cfg.EM_iters,
        err_tolerance=cfg.convergence_threshold,
        normalize_A=cfg.normalize_A,
        whiten_data = False,
        scale_source = False)

    results_EM_OLS_debias_iid[n_known] = ol_em.run_EM_OLS_debias_iid(outer_iters=100, outer_tol=cfg.convergence_threshold)(
        X=data.X,
        S_known=sknown,
        rng = np.random.default_rng(cfg.seed),
        EM_iter=cfg.EM_iters,
        err_tolerance=cfg.convergence_threshold,
        normalize_A_col=cfg.normalize_A,
        whiten_data = False,
        scale_source = False
        )
    
    results_EM_random[n_known] = ol_em.run_EM_random(
        X=data.X,
        S_known=sknown,
        rng = np.random.default_rng(cfg.seed),
        EM_iter=cfg.EM_iters,
        err_tolerance=cfg.convergence_threshold,
        whiten_data = False,
        scale_source = False
        )
    
    
    #results_fast_ICA[n_known] = ol_em.run_EM_FastICA(
    #    X=data.X,
    #    rng = np.random.default_rng(cfg.seed),
    #    S_known=sknown,
    #    EM_iter=cfg.EM_iters,
    #    err_tolerance=cfg.convergence_threshold,
    #    n_known=n_known)

    
comparison_runs = [
    (results_EM_OLS_iid, ol_plot.method_label('OLS_iid')),
    (results_EM_OLS_debias_iid, ol_plot.method_label('OLS_debias_iid')),
    #(results_EM_simple_wrong, ol_plot.method_label('standard_wrong')), 
    (results_EM_iid, ol_plot.method_label('standard_iid')),
    (results_sdp, ol_plot.method_label('OverICA')),
    (results_EM_random, ol_plot.method_label('random'))
]

n_known = 0
n_known = 1
n_known = 2
n_known = 3
n_known = 4
n_known = 5
n_known = 6


# VISUALIZE THE PROGRESS OF ESTIMATION FOR ONE DATASET
The plot below shows the different dimension of the data as blue dots. The green arrows are the true columns of A and the red arrows asre the estimated values. The orange arrows have been given by OLS.

Clicking the button gives one more known source. See the progress in the line plots at the bottom.


In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets
from ipywidgets import Button, HBox, IntSlider
from matplotlib.gridspec import GridSpec

# ===================================================================
# 1. High-Dimensional Grid Initialization (Executed ONCE)
# ===================================================================
max_val_X = np.abs(data.X).max()
max_val_A = np.abs(data.A).max()

arrow_scale = np.maximum(max_val_X, max_val_A)*0.9
A_true = ol_data.normalize_columns(data.A)
A_true_plot = A_true * arrow_scale
dsrc = A_true.shape[1]

global_plot_lim = np.maximum(max_val_X, max_val_A) * 1.1

# Quantify dimension footprint
n_features = data.X.shape[1]
n_rows = int(np.ceil(n_features / 2))
n_cols = len(comparison_runs)

# --- REPLACED SUBPLOTS WITH GRIDSPEC TO ALLOW SPANNING BOTTOM PLOT ---
fig = plt.figure(figsize=(3 * n_cols, 3 * n_rows + 8))
# Allocate n_rows for arrow projections + 1 extra row at the bottom for the error plot
gs = GridSpec(n_rows + 2, n_cols, figure=fig, height_ratios=[3] * n_rows + [4, 4])

# Generate an explicit 2D grid layout for arrow projections
axes = np.empty((n_rows, n_cols), dtype=object)

# Track arrows via a 3D nested collection layer: [row][column/method][atom]
all_true_arrows = []
all_est_arrows = []

for r in range(n_rows):
    row_true_refs = []
    row_est_refs = []
    
    # Identify target coordinate mappings for this row segment
    idx_x = 2 * r
    idx_y = 2 * r + 1
    has_y = idx_y < n_features
    label_y = f"x{idx_y + 1}" if has_y else "0"
    
    for c in range(n_cols):
        ax = fig.add_subplot(gs[r, c])
        ax.set_box_aspect(1)
        axes[r, c] = ax
        
        # Isolate coordinate slice arrays safely
        x_scatter = data.X[:, idx_x]
        y_scatter = data.X[:, idx_y] if has_y else np.zeros_like(x_scatter)
        
        # Render historical background point clouds
        ax.scatter(x_scatter, y_scatter, alpha=0.1, s=10, color='tab:blue', zorder=1)
        ax.set_aspect('equal')
        
        method_true_arrows = []
        method_est_arrows = []
        
        # Populate dummy coordinate tracks
        for i in range(cfg.nsrc):
            #t_arr = ax.arrow(0, 0, 0, 0, width=0.1, zorder=2)
            t_arr = ax.arrow(0, 0, 0, 0, width=0.05, head_width=0.3, zorder=2)
            #e_arr = ax.arrow(0, 0, 0, 0, width=0.2, ls='--', zorder=3)
            e_arr = ax.arrow(0, 0, 0, 0, width=0.05, head_width=0.3, ls='-', zorder=3)
            method_true_arrows.append(t_arr)
            method_est_arrows.append(e_arr)
            
        row_true_refs.append(method_true_arrows)
        row_est_refs.append(method_est_arrows)
        
        # Individual plot cosmetics
        ax.grid(True)
        ax.set_xlabel(f"x{idx_x + 1}")
        
        if c == 0:
            ax.set_ylabel(label_y)
            
        # If it's a 1D plot (x3 vs 0), fix the squishiness
        if not has_y:
            ax.set_ylim(-0.5, 0.5)
            ax.set_xlim(-global_plot_lim, global_plot_lim)
            ax.set_aspect('equal') 
        else:
            ax.set_ylim(-global_plot_lim, global_plot_lim)
            ax.set_xlim(-global_plot_lim, global_plot_lim)
            ax.set_aspect('equal', adjustable='box') # Keep data units perfectly 1:1 inside the square box
            
    all_true_arrows.append(row_true_refs)
    all_est_arrows.append(row_est_refs)
    
# ===================================================================
# FIXED: 1.5. Pre-compute and Render the Error Curves (Spanning Bottom)
# ===================================================================

# --- PLOT 1: Frobenius Error ---
ax_err = fig.add_subplot(gs[n_rows, :]) 
ax_err.set_title("Reconstruction Error vs. Number of Unknown Variables ($n_{unknown}$)", fontsize=11, fontweight='bold', pad=10)
ax_err.set_xlabel("Number of Unknown Variables ($n_{unknown}$)", fontsize=10)
ax_err.set_ylabel("Frobenius Error", fontsize=10)
ax_err.set_ylim(top=1.05, bottom=0)

ax_err_S = fig.add_subplot(gs[n_rows + 1, :]) # Handled via unique row index
ax_err_S.set_title("S MCC Error vs. Number of Unknown Variables ($n_{unknown}$)", fontsize=11, fontweight='bold', pad=10)
ax_err_S.set_xlabel("Number of Unknown Variables ($n_{unknown}$)", fontsize=10)
ax_err_S.set_ylabel("S MCC Error", fontsize=10)
ax_err_S.set_ylim(top = 1.05, bottom = 0)


markers_list = ['o', 's', '^', 'D', 'v', 'X', 'P', '*']

for idx, (results, method_name) in enumerate(comparison_runs):
    current_marker = markers_list[idx % len(markers_list)]
    u_axis = []
    err_axis = []
    u_S_axis = []
    err_S_axis = []
    for k in range(len(results)):
        out = results[k]
        if out is not None and 'A_est' in out:
            A_est_aligned, permutation, _, _, _ = ol_data.best_permutation_match_sign_flips(A=A_true, B=out['A_est'])
            
            err_val = ol_metric.frobenius_err(true=A_true, estimate=out['A_est'])[0]
            #np.cos(np.radians(ol_metric.matrix_angle_error(A_true=A_true, A_est=out['A_est'])['mean_error']))#
            sknown = ol_data.make_sknown(S=data.S, n_known_src=k)
            S_perm = out['S_est'][:, permutation].copy() 
            err_val_S = ol_metric.MCC_remove_known(true=data.S, estimate=S_perm, sknown=sknown)[0]
            
            u_axis.append(dsrc - k)
            err_axis.append(err_val)
            err_S_axis.append(err_val_S)
            
    sorted_indices = np.argsort(u_axis)
    ax_err.plot(np.array(u_axis)[sorted_indices], np.array(err_axis)[sorted_indices], marker=current_marker, label=method_name, alpha=0.7)
    ax_err_S.plot(np.array(u_axis)[sorted_indices], np.array(err_S_axis)[sorted_indices], marker=current_marker, label=method_name, alpha=0.7)

ax_err.axvline(x=n_features, color='black', linestyle='--', linewidth=1.5, label=f'Complete Limit ($n_{{unknown}} = {n_features}$)')
ax_err_S.axvline(x=n_features, color='black', linestyle='--', linewidth=1.5, label=f'Complete Limit ($n_{{unknown}} = {n_features}$)')

current_u_line_A = ax_err.axvline(x=dsrc, color='red', linestyle='-', linewidth=2.5, label='Current State')
current_u_line_S = ax_err_S.axvline(x=dsrc, color='red', linestyle='-', linewidth=2.5, label='Current State')

ax_err.grid(True)
ax_err.legend(loc='upper right', fontsize=9)

# FIXED: Targets are now correctly bound to ax_err_S
ax_err_S.grid(True)
ax_err_S.legend(loc='upper right', fontsize=9)

# Shared Global Legend Placement (Top)
legend_handles = [
    plt.Line2D([0], [0], color='tab:green', lw=3, label='True A (Unknown)'),
    plt.Line2D([0], [0], color='tab:orange', lw=3, label='Known Source / OLS'),
    plt.Line2D([0], [0], color='tab:red', lw=3, label='Estimated A (Unknown)')
]
fig.legend(handles=legend_handles, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.01))

# ===================================================================
# 2. Multi-Dimensional Synchronized Parallel Update Loop (With Stacking)
# ===================================================================
def update_all_plots(n_known):
    """
    Slices the high-dimensional mixing matrix projections across row pairs.
    Updates arrow coordinate layers alongside tracking indices down on the error plot.
    """
    current_u = dsrc - n_known

    # Visual Requirement 3: Update dynamic indicator line layout position on loop interactions
    current_u_line_A.set_xdata([current_u, current_u])
    current_u_line_S.set_xdata([current_u, current_u])
    
    
    for c, (results, method_name) in enumerate(comparison_runs):
        current_k = min(n_known, len(results) - 1)
        out = results[current_k]
        
        # Align global estimation coordinates to target structure
        A_est, _, _, _, _ = ol_data.best_permutation_match_sign_flips(
            A=A_true,
            B=out['A_est']
        )
        
        err = ol_metric.frobenius_err(true=A_true, estimate=out['A_est'])[0]
        #the true A has already had normalized columns - suffices to scale with arrow scale to get similar sized arrows
        A_est_plot = A_est * arrow_scale
        nobs, nsrc = A_est_plot.shape
        
        # Propagate changes downwards across row pairs
        for r in range(n_rows):
            ax = axes[r, c]
            idx_x = 2 * r
            idx_y = 2 * r + 1
            has_y = idx_y < n_features

            # Precompute unique vertical shelves if this row is 1D (e.g., x3 vs 0)
            if not has_y:
                y_positions = np.linspace(-0.35, 0.35, cfg.nsrc)

            for i in range(cfg.nsrc):
                if i < current_k:
                    alpha_val = 0.5        # Slightly transparent to let background show
                    col_true = 'tab:orange'
                    col_est = 'tab:orange'
                    
                    # Characteristic modifications for OLS: heavy line borders
                    all_est_arrows[r][c][i].set_linewidth(2.5)
                    all_est_arrows[r][c][i].set_linestyle('-')
                
                else:
                    # Blind EM estimation updates (i >= current_k)
                    alpha_val = 0.9        # High visibility for active learning targets
                    col_true = 'tab:green'
                    col_est = 'tab:red'
                    
                    # Characteristic modifications for Blind EM: standard thin borders
                    all_est_arrows[r][c][i].set_linewidth(1.0)
                    all_est_arrows[r][c][i].set_linestyle('--') # Dashed style remains for Blind entries
                
                
                # Extract horizontal vector lengths
                dx_t = A_true_plot[idx_x, i]
                dx_e = A_est_plot[idx_x, i]
                
                if has_y:
                    # Standard 2D Plot: All arrows anchor at (0,0)
                    dy_t = A_true_plot[idx_y, i]
                    dy_e = A_est_plot[idx_y, i]
                    
                    all_true_arrows[r][c][i].set_data(x=0.0, y=0.0, dx=dx_t, dy=dy_t)
                    all_est_arrows[r][c][i].set_data(x=0.0, y=0.0, dx=dx_e, dy=dy_e)
                else:
                    # 1D Plot: Stack arrows vertically by assigning unique base y-coordinates
                    y_pos = y_positions[i]
                    
                    all_true_arrows[r][c][i].set_data(x=0.0, y=y_pos, dx=dx_t, dy=0.0)
                    all_est_arrows[r][c][i].set_data(x=0.0, y=y_pos, dx=dx_e, dy=0.0)
                
                # Apply alpha and color updates
                all_true_arrows[r][c][i].set_alpha(alpha_val)
                all_true_arrows[r][c][i].set_color(col_true)
                all_est_arrows[r][c][i].set_alpha(alpha_val)
                all_est_arrows[r][c][i].set_color(col_est)

            # Title management
            if r == 0:
                ax.set_title(
                    f"Method: {method_name}\n"
                    f"n_obs = {nobs} |  n_u: {nsrc - current_k}  | Error: {np.round(err, 3)}",
                    fontsize=10, fontweight='bold'
                )
            else:
                coord_desc = f"Coords: x{idx_x + 1} & x{idx_y + 1}" if has_y else f"Stacked Coords: x{idx_x + 1} vs 0"
                ax.set_title(coord_desc, fontsize=9, color='gray')
                
    fig.canvas.draw_idle()
    
# ===================================================================
# 3. Component Control Interface Mount
# ===================================================================
n_known_slider = IntSlider(
    min=0,
    max=cfg.nsrc,
    step=1,
    value=0,
    description='Known Sources:',
    style={'description_width': 'initial'},
    continuous_update=True
)

btn_increment = Button(
    description='+1 Source All',
    disabled=False,
    button_style='info',
    icon='plus'
)

def on_btn_click(b):
    if n_known_slider.value < n_known_slider.max:
        n_known_slider.value += 1
    else:
        n_known_slider.value = 0

btn_increment.on_click(on_btn_click)

ui_layout = HBox([n_known_slider, btn_increment])
plot_output = ipywidgets.interactive_output(update_all_plots, {'n_known': n_known_slider})

# Render layout structures
display(ui_layout, plot_output)
fig.tight_layout(rect=[0, 0, 1, 0.96]) # Allocate spacing padding layer for top global legends

# Force initial execution framework alignment pass
update_all_plots(n_known_slider.value)

RuntimeError: 'widget' is not a recognised GUI loop or backend name